# STEP 2 — VIDEO GENERATION (Text/Image-to-Video)

## Useful Information

**Pricing**

Google Veo 3.1 Lite: 0.05 USD/sec → 3 USD/min

**Text-to-Video Generation**

Video Prompt  
↓  
Veo Model (`veo-3.1-lite-generate-preview`)  
↓  
Generated Video Clip

Text-to-video generation creates a video clip from a text prompt only. Each scene's `VideoPrompt` is sent to Veo, which returns an 8-second, 720p, 16:9 clip.

**Image-to-Video Generation**

Reference Image + Video Prompt  
↓  
Veo Model  
↓  
Generated Video Clip

Image-to-video generation uses a reference image as the starting frame, so the clip keeps the look of the image while the prompt describes how the scene should move.

**Asynchronous Generation (Polling)**

Submit Generation Job  
↓  
Receive Job ID  
↓  
Poll Job Status (every 10 seconds)  
↓  
Download Completed Video

Video generation takes minutes, so the API does not return the video immediately. It returns a job that the notebook checks every `POLL_INTERVAL_SECONDS` until it completes, and stops waiting after `GENERATION_TIMEOUT_SECONDS` (10 minutes).

## 1. Setup

### 1.1 Import

In [1]:
# Standard library
import os
import time
import json
from pathlib import Path
from enum import Enum
from typing import Optional
import random

# Environment variables
from dotenv import load_dotenv

# Data validation
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

# Google Veo
from google import genai
from google.genai import types

# LangGraph
from langgraph.graph import StateGraph, START, END

# Notebook display helpers
from IPython.display import Image, Video, display

### 1.2 Load Environment Variables

In [2]:
# Load environment variables from .env
load_dotenv()

AI_MODEL_API_KEY = os.getenv("AI_MODEL_API_KEY")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not AI_MODEL_API_KEY:
    raise ValueError("AI_MODEL_API_KEY is not set.")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not set.")

### 1.3 Create Output Directory

In [3]:
# Output directories
OUTPUT_DIR = Path("../outputs/video_generation")
VIDEO_DIR = OUTPUT_DIR / "videos"
METADATA_DIR = OUTPUT_DIR / "metadata"

# Create directories if they do not already exist
VIDEO_DIR.mkdir(parents = True, exist_ok = True)
METADATA_DIR.mkdir(parents = True, exist_ok = True)

print(f"Video directory: {VIDEO_DIR.resolve()}")
print(f"Metadata directory: {METADATA_DIR.resolve()}")

Video directory: /Users/yukpinglee/Desktop/agentic-video/outputs/video_generation/videos
Metadata directory: /Users/yukpinglee/Desktop/agentic-video/outputs/video_generation/metadata


### 1.4 Define Video Generation Settings

In [4]:
# Video generation
# Note: every Veo clip is VIDEO_DURATION_SECONDS long; the planned scene durations are not used.
# In the final video, the narration sets the length: the last frame is held if the scenes are shorter,
# and the video is trimmed if they are longer (nothing is time-stretched).
VIDEO_DURATION_SECONDS = 8 
VIDEO_RESOLUTION = "720p"
VIDEO_ASPECT_RATIO = "16:9"

# Polling
POLL_INTERVAL_SECONDS = 10          # check the asynchronous generation job every 10 seconds.
GENERATION_TIMEOUT_SECONDS = 600    # stop waiting after 10 minutes.

# Retry when Veo shared capacity is unavailable
MAX_RETRY_ATTEMPTS = 5
INITIAL_RETRY_DELAY_SECONDS = 10

# Veo generation configuration
video_config = types.GenerateVideosConfig(aspect_ratio = VIDEO_ASPECT_RATIO, resolution = VIDEO_RESOLUTION,)

## 2. Configure Video Generation Provider

### 2.1 Select Video Model

In [5]:
VIDEO_PROVIDER = "google"

VIDEO_MODEL = "veo-3.1-lite-generate-preview"

In [6]:
# Verify Google API credentials

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is missing. Add it to the .env file.")

print("Google API credentials configured.")

Google API credentials configured.


### 2.2 Create Provider Client

In [7]:
# Create Google Gen AI client
client = genai.Client(api_key = GEMINI_API_KEY)


## 3. Video Generation Models

### 3.1 Define GenerationStatus

In [8]:
class GenerationStatus(str, Enum):
    """Possible states of a video generation job."""

    PENDING = "pending"
    PROCESSING = "processing"
    COMPLETED = "completed"
    FAILED = "failed"

### 3.2 Define VideoGenerationRequest

In [9]:
class VideoGenerationRequest(BaseModel):
    """Request for generating one video scene."""

    scene_id: int = Field(description = "ID of the scene being generated")
    prompt: str = Field(description = "Prompt sent to the video generation model")

    reference_image: Optional[str] = Field(default = None, description = "Optional path to a reference image for image-to-video generation")  # default = None means if the user doesn't provide reference_image, automatically use None.

    duration_seconds: int = Field(default = VIDEO_DURATION_SECONDS, description = "Requested video duration in seconds")

    resolution: str = Field(default = VIDEO_RESOLUTION, description = "Requested video resolution")

    aspect_ratio: str = Field(default = VIDEO_ASPECT_RATIO, description = "Requested video aspect ratio")

### 3.3 Define GenerationResult

In [10]:
class GenerationResult(BaseModel):
    """Result of a video generation job."""

    scene_id: int = Field(description = "ID of the corresponding scene")

    job_id: Optional[str] = Field(default = None, description = "Provider generation job ID")

    status: GenerationStatus = Field(default = GenerationStatus.PENDING, description = "Current generation status")

    video_path: Optional[str] = Field(default = None, description = "Local path of the downloaded video")

    error_message: Optional[str] = Field(default = None, description = "Error message if generation fails")

## 4. Text-to-Video Generation

### 4.1 Submit Text-to-Video Generation

In [11]:
def submit_text_to_video(request: VideoGenerationRequest):
    """Submit a text-to-video generation request to Veo."""

    return client.models.generate_videos(model = VIDEO_MODEL,
                                            prompt = request.prompt,
                                            config = types.GenerateVideosConfig(duration_seconds = request.duration_seconds,
                                                                                resolution = request.resolution,
                                                                                aspect_ratio = request.aspect_ratio,
                                                                                )
                                        )

## 5. Image-to-Video Generation

### 5.1 Validate Reference Image

In [12]:
def validate_reference_image(image_path: str) -> Path:
    """Validate that the reference image exists and is a supported image file."""

    path = Path(image_path)

    if not path.exists():
        raise FileNotFoundError(f"Reference image not found: {path}")

    if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        raise ValueError("Reference image must be JPG, JPEG, or PNG.")

    return path

In [13]:
reference_image_path = validate_reference_image("reference_images/test_image.jpeg")

### 5.2 Prepare Reference Image

In [14]:
def prepare_reference_image(image_path: str) -> types.Image:
    """Load a local image in the format required by Veo."""

    path = validate_reference_image(image_path)

    image = types.Image.from_file(location = str(path))

    return image

In [15]:
reference_image = prepare_reference_image(reference_image_path)


### 5.3 Build Image-to-Video Request

In [16]:
# Veo 3.1 Lite only supports 1 image as the starting image
# Veo 3.1 Lite generates audio natively, and audio is always on. 
# The generated clip can contain ambient sound, sound effects, dialogue, and music when the prompt calls for it.
image_test_request = VideoGenerationRequest(scene_id = 2,
                                            prompt = ("AI doing research on human. "
                                                      "Natural cinematic movement, realistic lighting, smooth motion."
                                                      ),
                                            reference_image = str(reference_image_path)
                                            )

In [17]:
print(image_test_request)

scene_id=2 prompt='AI doing research on human. Natural cinematic movement, realistic lighting, smooth motion.' reference_image='reference_images/test_image.jpeg' duration_seconds=8 resolution='720p' aspect_ratio='16:9'


### 5.4 Submit Generation Job

In [18]:
def submit_image_to_video(request: VideoGenerationRequest):
    """Submit an image-to-video generation request to Veo."""

    if request.reference_image is None:
        raise ValueError("reference_image is required for image-to-video generation.")

    reference_image = prepare_reference_image(request.reference_image)

    operation = client.models.generate_videos(model = VIDEO_MODEL,
                                              prompt = request.prompt,
                                              image = reference_image,
                                              config = types.GenerateVideosConfig(duration_seconds = request.duration_seconds,
                                                                                  resolution = request.resolution,
                                                                                  aspect_ratio = request.aspect_ratio
                                                                                  ))

    return operation

## 6. Unified Video Generation 

Give the rest of your system one function for generating videos, regardless of whether an image is provided.

### 6.1 Generate Video

In [19]:
def generate_video(request: VideoGenerationRequest):
    """Submit a video generation request with retry for Veo capacity errors."""

    for attempt in range(MAX_RETRY_ATTEMPTS):
        try:
            if request.reference_image:
                return submit_image_to_video(request)

            return submit_text_to_video(request)

        except Exception as error:
            # Only retry Veo capacity errors
            if "RESOURCE_EXHAUSTED" not in str(error):
                raise

            # Stop after the final attempt
            if attempt == MAX_RETRY_ATTEMPTS - 1:   # If we've already reached the final allowed attempt, stop retrying and return the error
                raise

            # Exponential backoff with jitter
            wait_seconds = (INITIAL_RETRY_DELAY_SECONDS * (2 ** attempt) + random.uniform(0, 5))

            print(f"Veo capacity unavailable. "
                    f"Retrying in {wait_seconds:.1f} seconds...")

            time.sleep(wait_seconds)

## 7. Generation Status and Polling

### 7.1 Poll Generation

In [20]:
def poll_generation(operation):
    """Poll a Veo generation operation every POLL_INTERVAL_SECONDS until it completes."""

    start_time = time.time()

    while not operation.done:  # Stop if generation takes too long
        if time.time() - start_time > GENERATION_TIMEOUT_SECONDS:
            raise TimeoutError("Video generation timed out.")

        print("Video is still generating...")

        time.sleep(POLL_INTERVAL_SECONDS)

        operation = client.operations.get(operation)

    return operation

### 7.2 Validate Completed Generation

In [21]:
# Return the video clip once the status of the job changes to operation.done

def validate_completed_generation(operation):
    """Validate that the completed operation contains a generated video."""

    if not operation.done:  # operation.done is a property of the Veo generation operation
        raise ValueError("Generation operation is not complete.")

    if operation.response is None:
        raise RuntimeError("Generation completed without a response.")

    if not operation.response.generated_videos:
        raise RuntimeError("Generation completed without a generated video.")

    return operation.response.generated_videos[0]

### 7.3 Wait for Generation

In [22]:
# Purpose: Keep polling Google and get the returned video once operation.done

def wait_for_generation(operation):
    """Wait for a Veo generation job and return the generated video."""

    operation = poll_generation(operation)

    generated_video = validate_completed_generation(operation)

    return operation, generated_video

## 8. Download Generated Video

### 8.1 Build Video Output Path

In [23]:
def build_video_output_path(scene_id: int) -> Path:
    """Create the local output path for a generated scene."""

    return VIDEO_DIR / f"scene_{scene_id:03d}.mp4"

### 8.2 Download Generated Video

In [24]:
def download_generated_video(generated_video, output_path: Path) -> Path:
    """Download a generated Veo video to the local output directory."""

    client.files.download(file = generated_video.video, destination = str(output_path))

    return output_path

### 8.3 Validate Downloaded Video

In [25]:
def validate_video_file(video_path: Path) -> None:
    """Check that the downloaded video exists."""

    if not video_path.exists():
        raise FileNotFoundError(f"Downloaded video was not found: {video_path}")

    if video_path.stat().st_size == 0:
        raise ValueError(f"Downloaded video is empty: {video_path}")

### 8.4 Save Generated Video

In [26]:
def save_generated_video(generated_video, result: GenerationResult) -> GenerationResult:
    """Download, validate, and record a completed generated video."""

    output_path = build_video_output_path(result.scene_id)      # build the path

    download_generated_video(generated_video, output_path)      # download video

    validate_video_file(output_path)                            # validate if the video exists

    result.status = GenerationStatus.COMPLETED                  # change the GenerationStatus
    result.video_path = str(output_path)

    return result

## 9. Generate All Scenes

### 9.1 Build Generation Request

In [27]:
def build_generation_request(video_prompt, reference_image: Optional[str] = None) -> VideoGenerationRequest:
    """Convert a VideoPrompt into a video generation request."""

    # Note: every clip uses VIDEO_DURATION_SECONDS (8 s), not the planned scene duration.
    return VideoGenerationRequest(scene_id = video_prompt.scene_id,
                                  prompt = video_prompt.prompt,
                                  reference_image = reference_image,
                                  duration_seconds = VIDEO_DURATION_SECONDS,
                                  resolution = VIDEO_RESOLUTION,
                                  aspect_ratio = VIDEO_ASPECT_RATIO
                                  )

### 9.2 Generate One Scene

In [28]:
class VideoPrompt(BaseModel):

    scene_id: int = Field(description = "ID of the corresponding scene")
    prompt: str = Field(description = "Detailed prompt for the video-generation model")

In [29]:
def generate_scene(video_prompt: VideoPrompt, reference_image: Optional[str] = None) -> GenerationResult:

    request = build_generation_request(video_prompt, reference_image = reference_image)

    operation = generate_video(request)

    operation, generated_video = wait_for_generation(operation)

    result = GenerationResult(scene_id = video_prompt.scene_id,
                                status = GenerationStatus.PROCESSING
                            )

    return save_generated_video(generated_video, result)

### 9.3 Generate All Scenes

In [30]:
import time

def generate_all_scenes(video_prompts: list[VideoPrompt], reference_image: Optional[str] = None) -> list[GenerationResult]:
    """Generate video clips for all scenes."""

    results = []

    for index, video_prompt in enumerate(video_prompts):
        print(f"\nScene {index + 1}/{len(video_prompts)}")

        result = generate_scene(video_prompt, reference_image=reference_image)

        results.append(result)

        # Avoid exceeding Veo's 2 requests-per-minute limit
        if index < len(video_prompts) - 1:
            time.sleep(35)

    return results

### 9.4 Test Scene Generation

In [31]:
test_video_prompts = [VideoPrompt(scene_id = 62901,
                                  prompt = ("A cinematic wide shot of Sydney Harbour at sunrise. "
                                            "Warm golden light reflects across the water. "
                                            "The camera slowly moves forward. "
                                            "Natural realistic motion. "
                                            "No dialogue or narration."
                                            )),
                      VideoPrompt(scene_id = 62902,
                                  prompt = ("A cinematic view of the Sydney Opera House in warm morning light. "
                                            "People walk naturally along the harbour. "
                                            "The camera slowly pans across the scene. "
                                            "No dialogue or narration."
                                            ))]



### 9.5 Inspect Results

In [32]:
# PAID API CALL
# Starts one Veo generation per prompt.

generation_results = generate_all_scenes(video_prompts = test_video_prompts)



Scene 1/2


/var/folders/kw/xjs_s57d66dc3cnb0p3gwv180000gn/T/ipykernel_50167/2690844922.py:4: DeprecationWarning: The generate_videos method with prompt/image/video arguments is deprecated and will be removed in a future major release (not before 2026-07-31). Please use the source argument instead.
  return client.models.generate_videos(model = VIDEO_MODEL,


Video is still generating...
Video is still generating...
Video is still generating...
Video is still generating...

Scene 2/2
Video is still generating...
Video is still generating...
Video is still generating...
Video is still generating...


In [33]:
for result in generation_results:
    print(f"Scene {result.scene_id}: " f"{result.status.value}")

    if result.video_path:
        print(f"  Video: {result.video_path}")

    if result.error_message:
        print(f"  Error: {result.error_message}")

Scene 62901: completed
  Video: ../outputs/video_generation/videos/scene_62901.mp4
Scene 62902: completed
  Video: ../outputs/video_generation/videos/scene_62902.mp4
